# Chest X-ray 肺炎分類 — EfficientNet_B3 遷移學習（Google Colab 單檔版）

本 notebook 不需要任何外部檔案，整份貼進 Colab 由上往下執行即可。

- **Cell 0**：所有模組程式碼（constant / dataset / builder / trainer / visualize / utils）＋ 套件安裝
- **Cell 1**：環境設定（裝置偵測、掛載 Google Drive）
- **Cell 2**：實驗設定（換參數只改這裡，改完重跑 Cell 4+）
- **Cell 3**：下載資料（每次重開執行階段都要跑）
- **Cell 4**：準備資料集與模型
- **Cell 5**：訓練
- **Cell 6**：測試集評估
- **Cell 7**：訓練曲線與混淆矩陣
- **Cell 8**：儲存結果到 Drive

> 建議先在 Colab 選單設定 **執行階段 → 變更執行階段類型 → GPU**。

## 執行前準備

**1. Kaggle 憑證**（Cell 3 下載資料需要）

Kaggle → 右上頭像 → **Settings → API → Create New Token**，下載 `kaggle.json`，
裡面有 `username` 與 `key`。接著在 Colab 左側工具列點 **🔑 Secrets**，新增兩筆：

| Name | Value |
| --- | --- |
| `KAGGLE_USERNAME` | kaggle.json 的 `username` |
| `KAGGLE_KEY` | kaggle.json 的 `key` |

兩筆都要打開 **Notebook access**。沒設定的話 Cell 3 會改用 `kagglehub.login()`
跳出輸入框，手動貼上同樣兩個值也可以。

**2. Google Drive 輸出路徑**

Cell 1 會呼叫 `drive.mount("/content/drive")`（要按授權），Cell 2 的
`DRIVE_PROJECT_DIR` 決定結果存到哪，預設是
`我的雲端硬碟/Pneumonia_detection_model/outputs/<實驗名>_<時間戳>/`。


In [ ]:
# ============================================================================
# Cell 0 — 模組程式碼（原本 constant.py / data / models / engine / utils 的內容）
#          這個 Cell 只需執行一次，之後都不用再動。
# ============================================================================

# --- 套件安裝（Colab 用；本機若已安裝可忽略） --------------------------------
# Colab 預設已有 torch / torchvision / matplotlib / numpy，只缺 kagglehub。
!pip install -q kagglehub

# --- 標準與第三方套件 --------------------------------------------------------
import copy
import json
import os
import random
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import (
    efficientnet_b3,
    EfficientNet_B3_Weights,
    resnet18,
    ResNet18_Weights,
    resnet50,
    ResNet50_Weights,
)


# ============================================================================
# constant.py
# ============================================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODELS = {
    "EFFICIENTNET_B3": "efficientnet_b3",
    "RESNET18": "resnet18",
    "RESNET50": "resnet50",
}
DEFAULT_MODEL_NAME = MODELS["EFFICIENTNET_B3"]

POSITIVE_CLASS_NAME = "PNEUMONIA"


# ============================================================================
# data/dataset.py
# ============================================================================
KAGGLE_DATASET_ID = "paultimothymooney/chest-xray-pneumonia"


def in_colab():
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def ensure_kaggle_auth():
    """確保 kagglehub 有 Kaggle 憑證可用，回傳憑證來源字串。

    嘗試順序：
      1. 環境變數 KAGGLE_USERNAME / KAGGLE_KEY
      2. Colab Secrets（左側 🔑 圖示，新增同名兩個 secret 並開啟 Notebook access）
      3. ~/.kaggle/kaggle.json
      4. kagglehub.login() 互動式輸入

    憑證取得：Kaggle → 右上頭像 → Settings → API → Create New Token，
    下載的 kaggle.json 裡就是 username 與 key。
    """
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return "環境變數"

    if in_colab():
        try:
            from google.colab import userdata

            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            return "Colab Secrets"
        except Exception as e:
            print(f"未能從 Colab Secrets 取得憑證（{type(e).__name__}），改用互動式登入。")

    if (Path.home() / ".kaggle" / "kaggle.json").exists():
        return "~/.kaggle/kaggle.json"

    import kagglehub

    kagglehub.login()  # 會跳出輸入框，依序填 username 與 key
    return "互動式登入"


def find_split_root(root):
    """在下載目錄底下找出真正含 train/val/test 的那一層。

    Kaggle 上這個資料集不同版本會多包一層 chest_xray/，所以不寫死路徑。
    """
    root = Path(root)
    candidates = [root, root / "chest_xray", root / "chest_xray" / "chest_xray"]
    candidates += sorted(root.glob("*/chest_xray"))

    for candidate in candidates:
        if all((candidate / split).is_dir() for split in ("train", "val", "test")):
            return candidate

    raise FileNotFoundError(f"在 {root} 底下找不到同時含 train/val/test 的資料夾")


def download_dataset(local_dir=None):
    """下載 chest-xray-pneumonia，回傳含 train/val/test 的資料夾路徑。

    local_dir 指定且已有有效資料時，直接沿用不重新下載
    （例如事先解壓到 Drive 的副本）。
    """
    if local_dir is not None:
        try:
            split_root = find_split_root(local_dir)
            print(f"沿用既有資料：{split_root}")
            return split_root
        except FileNotFoundError:
            print(f"{local_dir} 沒有可用資料，改從 Kaggle 下載。")

    print(f"Kaggle 憑證來源：{ensure_kaggle_auth()}")

    import kagglehub

    cache_root = Path(kagglehub.dataset_download(KAGGLE_DATASET_ID))
    print(f"kagglehub 快取路徑：{cache_root}")
    return find_split_root(cache_root)


def build_transforms(img_size=224):
    train_tf = transforms.Compose(
        [
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(size=img_size, scale=(0.85, 1.0)),
            transforms.RandomRotation(degrees=7),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )

    eval_tf = transforms.Compose(
        [
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )

    return {"train": train_tf, "val": eval_tf, "test": eval_tf}


def build_datasets(dataset_root, img_size=224):
    """Build train/val/test ImageFolder datasets with consistent class_to_idx."""
    dataset_root = Path(dataset_root)
    tf = build_transforms(img_size=img_size)

    image_datasets = {}
    for split in ["train", "val", "test"]:
        split_dir = dataset_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"資料夾不存在: {split_dir}")
        image_datasets[split] = ImageFolder(root=str(split_dir), transform=tf[split])

    train_class_to_idx = image_datasets["train"].class_to_idx
    for split in ["val", "test"]:
        if image_datasets[split].class_to_idx != train_class_to_idx:
            raise ValueError(
                f"{split} 類別索引與 train 不一致: {image_datasets[split].class_to_idx} != {train_class_to_idx}"
            )

    idx_to_class = {v: k for k, v in train_class_to_idx.items()}
    return image_datasets, train_class_to_idx, idx_to_class


def make_dataloaders(datasets, batch_size, num_workers=2):
    return {
        split: DataLoader(
            datasets[split],
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
        for split in datasets
    }


# ============================================================================
# models/builder.py
# ============================================================================
# model name -> (constructor, pretrained weights)
_MODEL_REGISTRY = {
    MODELS["EFFICIENTNET_B3"]: (efficientnet_b3, EfficientNet_B3_Weights.IMAGENET1K_V1),
    MODELS["RESNET18"]: (resnet18, ResNet18_Weights.IMAGENET1K_V1),
    MODELS["RESNET50"]: (resnet50, ResNet50_Weights.IMAGENET1K_V2),
}


def replace_head(model, num_classes):
    """Replace the final Linear layer with a new one sized for num_classes."""
    last_layer_name, last_layer = list(model.named_modules())[-1]
    parent = model
    for part in last_layer_name.split(".")[:-1]:
        parent = getattr(parent, part)
    attr = last_layer_name.split(".")[-1]
    setattr(parent, attr, nn.Linear(last_layer.in_features, num_classes))
    return model


def build_model(num_classes, device, model_name=DEFAULT_MODEL_NAME, pretrained=True, freeze_backbone=False):
    """Return a model with a custom classification head on device."""
    if model_name not in _MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{model_name}'. Available: {list(_MODEL_REGISTRY)}")
    model_fn, weights = _MODEL_REGISTRY[model_name]
    model = model_fn(weights=weights if pretrained else None)
    model = replace_head(model, num_classes)

    if freeze_backbone:
        for name, param in model.named_parameters():
            if "classifier" not in name and "fc" not in name:
                param.requires_grad = False

    return model.to(device)


# ============================================================================
# engine/trainer.py
# ============================================================================
def compute_class_weights(dataset, num_classes, device):
    targets = torch.tensor(dataset.targets, dtype=torch.long)
    counts = torch.bincount(targets, minlength=num_classes).float()
    counts = torch.clamp(counts, min=1.0)
    weights = counts.sum() / (num_classes * counts)
    return weights.to(device)


def binary_metrics_from_logits(logits, labels, positive_index=1):
    preds = logits.argmax(dim=1)

    tp = ((preds == positive_index) & (labels == positive_index)).sum().item()
    tn = ((preds != positive_index) & (labels != positive_index)).sum().item()
    fp = ((preds == positive_index) & (labels != positive_index)).sum().item()
    fn = ((preds != positive_index) & (labels == positive_index)).sum().item()
    return tp, tn, fp, fn


def summarize_metrics(loss_sum, samples, tp, tn, fp, fn):
    loss = loss_sum / max(samples, 1)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (2 * precision * recall) / max(precision + recall, 1e-8)

    return {
        "loss": loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def run_one_epoch(model, device, loader, criterion, optimizer=None, positive_index=1):
    """
    Run one training or evaluation epoch.

    Pass optimizer=None for eval mode (no gradient updates).
    Returns a dict with loss, accuracy, precision, recall, f1, tp/tn/fp/fn, cm.
    """
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_samples = 0
    total_tp = total_tn = total_fp = total_fn = 0
    cm = torch.zeros(2, 2, dtype=torch.long)

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        preds = logits.detach().argmax(dim=1)
        tp, tn, fp, fn = binary_metrics_from_logits(logits.detach(), labels, positive_index=positive_index)
        total_tp += tp
        total_tn += tn
        total_fp += fp
        total_fn += fn

        # true_idx*2 + pred_idx -> flattened index into a 2x2 matrix, independent of positive_index
        flat = (labels.cpu() * 2 + preds.cpu()).long()
        cm += torch.bincount(flat, minlength=4).reshape(2, 2)

    metrics = summarize_metrics(total_loss, total_samples, total_tp, total_tn, total_fp, total_fn)
    metrics["cm"] = cm.numpy()
    return metrics


def train_model(
    model,
    device,
    criterion,
    optimizer,
    scheduler,
    dataloaders,
    epochs=12,
    unfreeze_epoch=4,
    lr=1e-3,
    backbone_lr_factor=0.1,
    should_freeze_backbone=True,
    positive_index=1,
    learning_depend_metric="f1",
    epoch_callback=None,
    early_stopping=True,
    patience=6,
    min_delta=1e-4,
    checkpoint_guard=None,
):
    """
    Full training loop with optional backbone unfreeze.

    When should_freeze_backbone=True, the model's backbone should already be
    frozen and the optimizer should contain only classifier params before this
    call. At unfreeze_epoch, backbone params are unfrozen and added as a
    second param group with lr * backbone_lr_factor.
    """
    best_state = None
    monitor_mode = "min" if "loss" in learning_depend_metric else "max"
    best_score = float("inf") if monitor_mode == "min" else float("-inf")
    best_cm = None
    history = []
    patience_counter = 0

    for epoch in range(epochs):
        print(f"===== Epoch {epoch + 1}/{epochs} =====")

        if epoch == unfreeze_epoch and should_freeze_backbone:
            print("===== Unfreezing Backbone =====")
            backbone_params = []
            for name, param in model.named_parameters():
                if "classifier" not in name and not param.requires_grad:
                    param.requires_grad = True
                    backbone_params.append(param)
            if backbone_params:
                optimizer.add_param_group({"params": backbone_params, "lr": lr * backbone_lr_factor})

        train_m = run_one_epoch(model, device, dataloaders["train"], criterion, optimizer, positive_index)
        val_m = run_one_epoch(model, device, dataloaders["val"], criterion, None, positive_index)

        scheduler.step(val_m[learning_depend_metric])

        history.append({
            "epoch": epoch,
            "train_loss": train_m["loss"],
            "val_loss": val_m["loss"],
            "train_f1": train_m["f1"],
            "val_f1": val_m["f1"],
            "train_acc": train_m["accuracy"],
            "val_acc": val_m["accuracy"],
            "train_precision": train_m["precision"],
            "val_precision": val_m["precision"],
            "train_recall": train_m["recall"],
            "val_recall": val_m["recall"],
            "lr": optimizer.param_groups[0]["lr"],
        })

        print_metrics(train_m, val_m)

        if epoch_callback is not None:
            epoch_callback(val_m, history, epoch)

        score = val_m[learning_depend_metric]
        is_best = (score < best_score - min_delta) if monitor_mode == "min" else (score > best_score + min_delta)
        if is_best and checkpoint_guard is not None:
            is_best = checkpoint_guard(val_m, history, epoch)
        if is_best:
            best_score = score
            best_cm = val_m["cm"]
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping and patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch + 1} (no improvement for {patience} epochs)")
            break

    return best_state, best_score, best_cm, history


def print_metrics(train_m, val_m):
    header = f"{'Split':<6} | {'Loss':>6} | {'F1':>6} | {'Acc':>6} | {'Precision':>9} | {'Recall':>6}"
    print(header)
    print("-" * len(header))
    for name, m in [("Train", train_m), ("Val", val_m)]:
        print(
            f"{name:<6} | {m['loss']:6.4f} | {m['f1']:6.4f} | {m['accuracy']:6.4f} | "
            f"{m['precision']:9.4f} | {m['recall']:6.4f}"
        )


# ============================================================================
# engine/visualize.py
# ============================================================================
def plot_history(history, show=False):
    """Plot and return a 2×3 figure of training curves."""
    epochs = [h["epoch"] for h in history]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    pairs = [
        (axes[0, 0], "Loss", "train_loss", "val_loss"),
        (axes[0, 1], "F1 Score", "train_f1", "val_f1"),
        (axes[0, 2], "Accuracy", "train_acc", "val_acc"),
        (axes[1, 0], "Precision", "train_precision", "val_precision"),
        (axes[1, 1], "Recall", "train_recall", "val_recall"),
    ]
    for ax, title, train_key, val_key in pairs:
        ax.plot(epochs, [h[train_key] for h in history], marker="o", label="Train")
        ax.plot(epochs, [h[val_key] for h in history], marker="s", label="Val")
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True)

    ax = axes[1, 2]
    ax.plot(epochs, [h["lr"] for h in history], marker="D", color="tab:purple", label="LR")
    ax.set_title("Learning Rate", fontsize=14)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True)

    for row in axes:
        for a in row:
            a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    fig.suptitle("Training History", fontsize=16, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if show:
        plt.show()
    return fig


def plot_cm(cm, class_names=None, show=False):
    """Plot and return a row-normalised confusion matrix figure."""
    cm = np.asarray(cm)
    num_classes = cm.shape[0]
    labels = class_names if class_names is not None else list(range(num_classes))

    cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap=plt.cm.Blues, vmin=0, vmax=1)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(num_classes),
        yticks=np.arange(num_classes),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted Label",
        ylabel="True Label",
        title="Confusion Matrix (Row-Normalised)",
    )

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(
                j, i,
                f"{cm_norm[i, j]:.2f}\n({cm[i, j]})",
                ha="center", va="center", fontsize=9,
                color="white" if cm_norm[i, j] > 0.5 else "black",
            )

    fig.tight_layout()
    if show:
        plt.show()
    return fig


# ============================================================================
# utils.py
# ============================================================================
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class EpochTimer:
    def __init__(self):
        self._times = []
        self._last = time.time()

    def __call__(self, val_m, history, epoch):
        now = time.time()
        self._times.append(now - self._last)
        self._last = now

    def get_time_per_epoch(self):
        if not self._times:
            return 0.0
        avg = sum(self._times) / len(self._times)
        print(f"Average time per epoch: {avg:.1f}s")
        return avg


def make_output_dir(experiment_name, base="outputs"):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = Path(base) / f"{experiment_name}_{stamp}"
    out.mkdir(parents=True, exist_ok=True)
    return out


def save_results(
    out_dir,
    model,
    best_state,
    best_val_f1,
    test_metrics,
    class_to_idx,
    idx_to_class,
    img_size,
    history,
    model_name=DEFAULT_MODEL_NAME,
    train_time_per_epoch=0.0,
):
    model.load_state_dict(best_state)

    num_parameters = sum(p.numel() for p in model.parameters()) / 1e6  # 單位：M

    torch.save(best_state, out_dir / "best_model.pth")

    metadata = {
        "model_name": model_name,
        "num_parameters": num_parameters,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "img_size": img_size,
        "best_val_f1": best_val_f1,
        "test_metrics": {k: v for k, v in test_metrics.items() if k != "cm"},
        "train_time_per_epoch": train_time_per_epoch,
    }
    with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    with open(out_dir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    history_fig = plot_history(history)
    history_fig.savefig(out_dir / "training_curves.png", dpi=150)
    plt.close(history_fig)

    if test_metrics.get("cm") is not None:
        class_names = [idx_to_class[0], idx_to_class[1]]
        cm_fig = plot_cm(test_metrics["cm"], class_names=class_names)
        cm_fig.savefig(out_dir / "confusion_matrix.png", dpi=150)
        plt.close(cm_fig)

    print(f"Results saved to {out_dir}")


print("Cell 0 完成：模組已載入")

In [ ]:
# Cell 1 — 環境設定與 Google Drive 掛載（只需執行一次）
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用裝置: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- 掛載 Google Drive ------------------------------------------------------
# 執行後會跳出授權視窗，允許之後 Drive 就會出現在 /content/drive。
# /content/drive/MyDrive 對應雲端硬碟的「我的雲端硬碟」根目錄。
DRIVE_ROOT = Path("/content/drive/MyDrive")
drive_mounted = False

if in_colab():
    from google.colab import drive as colab_drive

    colab_drive.mount("/content/drive")  # 已掛載時會直接略過
    drive_mounted = DRIVE_ROOT.is_dir()
    print(f"Drive 掛載: {drive_mounted}")
else:
    print("非 Colab 環境，略過 Drive 掛載")


In [ ]:
# Cell 2 — 實驗設定（換參數只改這裡，改完重跑 Cell 4+）
EXPERIMENT_NAME     = "efficientnet_b3"
MODEL_NAME          = DEFAULT_MODEL_NAME
IMAGE_SIZE          = 224
BATCH_SIZE          = 32
EPOCHS              = 12
LR                  = 1e-3
WEIGHT_DECAY        = 1e-4
RANDOM_SEED         = 42

FREEZE_BACKBONE     = True   # 前幾個 epoch 先只訓練分類頭
UNFREEZE_EPOCH      = 4      # 於此 epoch 解凍 backbone
BACKBONE_LR_FACTOR  = 0.1    # 解凍後 backbone 學習率 = LR * factor

# --- 路徑設定 ---------------------------------------------------------------
# 想換到別的資料夾就改這一行，中間層不存在會自動建立。
DRIVE_PROJECT_DIR   = DRIVE_ROOT / "Pneumonia_detection_model"

# 輸出（best_model.pth / metadata.json / history.json / 兩張圖）寫到 Drive，
# 執行階段結束才不會一起消失；沒掛載 Drive 時退回本機 outputs/。
OUTPUT_BASE         = str(DRIVE_PROJECT_DIR / "outputs") if drive_mounted else "outputs"

# 資料集留在 Colab 本機磁碟（5,800 個小檔，走 Drive 會嚴重拖慢訓練）。
# 若已事先把 chest_xray 解壓到 Drive 想直接沿用，改成：
# DATASET_LOCAL_DIR = DRIVE_PROJECT_DIR / "data"
DATASET_LOCAL_DIR   = None

print(f"輸出根目錄: {OUTPUT_BASE}")
set_seed(RANDOM_SEED)


In [ ]:
# Cell 3 — 下載資料（每次重開執行階段都要跑；已下載過會走 kagglehub 快取）
dataset_root = download_dataset(local_dir=DATASET_LOCAL_DIR)
print(f"資料集路徑: {dataset_root}\n")

for split in ["train", "val", "test"]:
    counts = {
        d.name: sum(1 for p in d.iterdir() if p.is_file())
        for d in sorted((dataset_root / split).iterdir())
        if d.is_dir()
    }
    print(f"{split:<5} {counts}")


In [ ]:
# Cell 4 — 準備資料集與模型
image_datasets, class_to_idx, idx_to_class = build_datasets(dataset_root, img_size=IMAGE_SIZE)
num_classes = len(class_to_idx)
positive_index = class_to_idx.get(POSITIVE_CLASS_NAME, 1)
print(f"類別: {class_to_idx}, 正類別索引({POSITIVE_CLASS_NAME}): {positive_index}")

dataloaders = make_dataloaders(image_datasets, BATCH_SIZE)

model = build_model(
    num_classes, device,
    model_name=MODEL_NAME,
    freeze_backbone=FREEZE_BACKBONE,
)

class_weights = compute_class_weights(image_datasets["train"], num_classes, device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
print(f"模型與資料集準備完成，類別數：{num_classes}")

In [ ]:
# Cell 5 — 訓練
timer = EpochTimer()

best_state, best_val_f1, best_cm, history = train_model(
    model, device, criterion, optimizer, scheduler, dataloaders,
    epochs=EPOCHS,
    unfreeze_epoch=UNFREEZE_EPOCH,
    lr=LR,
    backbone_lr_factor=BACKBONE_LR_FACTOR,
    should_freeze_backbone=FREEZE_BACKBONE,
    positive_index=positive_index,
    epoch_callback=timer,
    early_stopping=True,
    patience=6,
)
print(f"\n訓練完成，最佳驗證 F1：{best_val_f1:.4f}")

time_per_epoch = timer.get_time_per_epoch()

In [ ]:
# Cell 6 — 測試集評估
model.load_state_dict(best_state)
test_m = run_one_epoch(model, device, dataloaders["test"], criterion, positive_index=positive_index)

print(
    "\n[Test] "
    f"loss={test_m['loss']:.4f}, acc={test_m['accuracy']:.4f}, "
    f"precision={test_m['precision']:.4f}, recall={test_m['recall']:.4f}, f1={test_m['f1']:.4f}"
)

In [ ]:
# Cell 7 — 訓練曲線與混淆矩陣
plot_history(history, show=True)
plot_cm(test_m["cm"], class_names=[idx_to_class[0], idx_to_class[1]], show=True)

In [ ]:
# Cell 8 — 儲存結果
out_dir = make_output_dir(EXPERIMENT_NAME, base=OUTPUT_BASE)
save_results(
    out_dir, model, best_state, best_val_f1, test_m,
    class_to_idx, idx_to_class, IMAGE_SIZE, history,
    model_name=MODEL_NAME, train_time_per_epoch=time_per_epoch,
)
print(f"結果已儲存至：{out_dir}")